# Session 6 — Real case: Electronics replenishment exceptions

**Salih Tutun, PhD** · DAT 5668 · Module 3 · Explain → solve a real desk (~80 min)

## The real problem (not a toy)

You own the **Electronics** replenishment desk.  
Real operations log: `inventory_dataset_full.csv` (**730 Electronics days**) with:

| Column | Business meaning |
| --- | --- |
| `Current_Inventory` | Morning stock (state) |
| `Demand` / `Units_Sold` / `Stockout_Units` | Customers vs service |
| `Action_Taken` | `Order` / `Maintain` / `Reduce` |
| `Order_Quantity` / `Order_Placed` | Replenishment details |
| `Holding_Cost` / `Stockout_Cost` / `Profit` | Money / reward language |
| `Is_Promotional` | Campaign day — not only a stock number |
| `Lead_Time` / `Reorder_Point` | Supply constraints |

**Job:** who owns the **daily numeric order** vs **promo/exception overrides**?  
Memo: `LLM` / `RL` / `hybrid` / `do not automate`.

![Two families](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-two-families.png)
![Hybrid](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)

*Images from GitHub.*

| Half | What we do |
| --- | --- |
| **I. Topic** | Explain the real case |
| **II. Practice** | Load real log → decide on real days → pass memo |


# Part I — Topic

## 1. Same loop — real money

![Loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/agent-loop.png)
![Inventory loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/session05-loop.png)

```text
observe inventory s → decide a → demand → profit/holding/stockout → s'
```

## 2. Why both families appear in THIS log

![Numeric path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

- **RL/MDP path:** `Current_Inventory`, `Demand`, `Order_Quantity`, `Profit`, costs — every day.
- **LLM + human path:** `Is_Promotional` days need a campaign brief / override ticket — a **document**.

![Document path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-llm-agent.png)
![Stamp](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-handoff.png)

**Teach aloud:** proposal ≠ release. Stock number ≠ promo brief.

## 3. How to choose

![Families](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/two-families.png)
![Wrong](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/wrong-family.png)
![ML tool](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/ml-as-tool.png)

| Situation | Lean toward |
| --- | --- |
| Daily inventory → order with profit/costs | **RL** |
| Promo/supplier documents | **LLM** (or person) |
| This Electronics log (both) | **Hybrid** |
| Reward unsigned / exploration unsafe | **Do not automate** |

Forecasts of `Demand` are **tools**, not the agent.

## 4. What “solve” means tonight

![Target](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)

1. Load Electronics rows.  
2. Measure promo rate, stockouts, profits.  
3. Open a **real date**.  
4. Basestock action from inventory.  
5. If promotional: draft override for human stamp.  
6. Memo: who **releases** order vs override.

| Cue | Do |
| --- | --- |
| **OPEN / RUN WITH ME / TRY THIS / CHECK / LAB** | Same as Session 5 |

**Stay in Topic until I say OPEN Stop 0.**

---
# Part II — Practice on the real Electronics desk


## Stop 0 — OPEN · find `inventory_dataset_full.csv`

**RUN WITH ME.**


In [ ]:
# ============================================================
# STOP 0 — SETUP · RUN WITH ME
# Real case file: inventory_dataset_full.csv
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/DAT5668/data")
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted.")
except ImportError:
    print("Not Colab — OK. Local data is fine.")

if DRIVE_ROOT.exists():
    files = sorted(p.name for p in DRIVE_ROOT.glob("*.csv"))
    print("CSVs:", files)
    print("inventory_dataset_full.csv →", "YES" if "inventory_dataset_full.csv" in files else "MISSING")
else:
    print("Drive folder not found yet (OK if local):", DRIVE_ROOT)


## Stop 1 — Load the real desk

![Desk](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-mdp.png)

**1a RUN WITH ME** · **1b TRY THIS** from the printout.


In [ ]:
# ============================================================
# STOP 1a — LOAD THE REAL ELECTRONICS DESK · RUN WITH ME
# ============================================================
from pathlib import Path
from collections import Counter
import csv
import statistics

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
PRODUCT = "Electronics"


def course_data(name: str) -> Path:
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("../../Contents/Module 4"),
        Path("../Contents/Module 4"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            print("Using:", path)
            return path
    raise FileNotFoundError("Could not find " + name + " — put it in MyDrive/DAT5668/data/")


def _flag(val) -> bool:
    return str(val).strip() in {"1", "True", "true", "YES", "Yes"}


def load_electronics_desk():
    rows = []
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product") or "").strip() != PRODUCT:
                continue
            rows.append(
                {
                    "Date": row["Date"],
                    "Current_Inventory": int(float(row["Current_Inventory"])),
                    "Demand": int(float(row["Demand"])),
                    "Units_Sold": int(float(row["Units_Sold"])),
                    "Stockout_Units": int(float(row["Stockout_Units"])),
                    "Lead_Time": int(float(row["Lead_Time"])),
                    "Price": float(row["Price"]),
                    "Is_Promotional": _flag(row.get("Is_Promotional", 0)),
                    "Is_Weekend": _flag(row.get("Is_Weekend", 0)),
                    "Action_Taken": (row.get("Action_Taken") or "").strip(),
                    "Order_Quantity": int(float(row.get("Order_Quantity") or 0)),
                    "Order_Placed": _flag(row.get("Order_Placed", 0)),
                    "Holding_Cost": float(row.get("Holding_Cost") or 0),
                    "Stockout_Cost": float(row.get("Stockout_Cost") or 0),
                    "Profit": float(row.get("Profit") or 0),
                    "Revenue": float(row.get("Revenue") or 0),
                    "Season": (row.get("Season") or "").strip(),
                    "Reorder_Point": int(float(row.get("Reorder_Point") or 0)),
                }
            )
    if not rows:
        raise ValueError("No Electronics rows found.")
    rows.sort(key=lambda r: r["Date"])
    return rows


ROWS = load_electronics_desk()
N_DAYS = len(ROWS)
AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)
AVG_INV = statistics.mean(r["Current_Inventory"] for r in ROWS)
PROMO_DAYS = [r for r in ROWS if r["Is_Promotional"]]
STOCKOUT_DAYS = [r for r in ROWS if r["Stockout_Units"] > 0]
ACTIONS = Counter(r["Action_Taken"] for r in ROWS)

print("=== REAL CASE: Electronics replenishment desk ===")
print("Days in log:", N_DAYS)
print("Avg inventory:", round(AVG_INV, 1), "| Avg demand:", round(AVG_DEMAND, 1))
print("Promo days:", len(PROMO_DAYS), f"({100*len(PROMO_DAYS)/N_DAYS:.1f}%)")
print("Stockout days:", len(STOCKOUT_DAYS), f"({100*len(STOCKOUT_DAYS)/N_DAYS:.1f}%)")
print("Actions in data:", dict(ACTIONS))
print("Mean profit/day: $", round(statistics.mean(r["Profit"] for r in ROWS), 1))
print("Mean holding cost/day: $", round(statistics.mean(r["Holding_Cost"] for r in ROWS), 1))
print("Mean stockout cost/day: $", round(statistics.mean(r["Stockout_Cost"] for r in ROWS), 1))
print("Reorder point in data:", ROWS[0]["Reorder_Point"])
print("Sample row:", {k: ROWS[10][k] for k in ["Date", "Current_Inventory", "Demand", "Stockout_Units", "Is_Promotional", "Action_Taken", "Profit"]})

demands = sorted(r["Demand"] for r in ROWS)
q1, q2 = demands[len(demands) // 3], demands[(2 * len(demands)) // 3]


def demand_bin(d: int) -> int:
    if d <= q1:
        return 0
    if d <= q2:
        return 1
    return 2


DEMAND_P = Counter(demand_bin(r["Demand"]) for r in ROWS)
DEMAND_P = {k: DEMAND_P[k] / N_DAYS for k in (0, 1, 2)}
print("Demand bins P(d):", {k: round(v, 3) for k, v in DEMAND_P.items()}, "| cuts", q1, q2)

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    axes[0].bar(["0", "1", "2"], [DEMAND_P[0], DEMAND_P[1], DEMAND_P[2]], color="#2F5D50")
    axes[0].set_title("Real Electronics demand bins")
    axes[0].set_xlabel("d"); axes[0].set_ylabel("P(d)"); axes[0].set_ylim(0, 1)
    axes[1].bar(["promo", "no promo"], [len(PROMO_DAYS), N_DAYS - len(PROMO_DAYS)], color=["#C45C26", "#2F5D50"])
    axes[1].set_title("Real promotional days")
    plt.tight_layout(); plt.show()


In [ ]:
# STOP 1b — TRY THIS
n_days_guess = None        # <<< FILL Days in log
promo_count_guess = None   # <<< FILL Promo days
has_profit_column = None   # <<< FILL True/False
print(n_days_guess, promo_count_guess, has_profit_column)
if None in (n_days_guess, promo_count_guess, has_profit_column):
    print("NOT YET")
else:
    ok = n_days_guess == N_DAYS and promo_count_guess == len(PROMO_DAYS) and has_profit_column is True
    print("CHECK 1 —", "PASS" if ok else "NOT YET")


## Stop 2 — Map real columns → state/action

![MDP](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

| Data `Action_Taken` | $a$ |
| --- | --- |
| Reduce | 0 |
| Maintain | 1 |
| Order | 2 |


In [ ]:
# STOP 2a — RUN WITH ME

def observe_state(inventory, avg_demand=AVG_DEMAND):
    inv = float(inventory)
    if inv < 0.75 * avg_demand:
        return "Low"
    if inv < 1.5 * avg_demand:
        return "Medium"
    return "High"

def action_to_code(action_taken: str) -> int:
    return {"Reduce": 0, "Maintain": 1, "Order": 2}.get(action_taken, -1)

def decide_basestock(state: str) -> int:
    return {"Low": 2, "Medium": 1, "High": 0}[state]

print("avg demand =", round(AVG_DEMAND, 1))
for r in ROWS[0:5]:
    s = observe_state(r["Current_Inventory"])
    print(r["Date"], "inv=", r["Current_Inventory"], "→", s, "| data=", r["Action_Taken"], action_to_code(r["Action_Taken"]), "| basestock=", decide_basestock(s))


In [ ]:
# STOP 2b — TRY THIS
inv_A = ROWS[0]["Current_Inventory"]
inv_B = int(AVG_DEMAND)
inv_C = int(2.5 * AVG_DEMAND)
state_A = ""  # <<< FILL Low/Medium/High
state_B = ""  # <<< FILL
state_C = ""  # <<< FILL
a_A = None; a_B = None; a_C = None  # <<< FILL 0/1/2
print(inv_A, state_A, a_A); print(inv_B, state_B, a_B); print(inv_C, state_C, a_C)
if not state_A or not state_B or not state_C or None in (a_A, a_B, a_C):
    print("NOT YET")
else:
    ok = (state_A, a_A) == (observe_state(inv_A), decide_basestock(observe_state(inv_A))) and (state_B, a_B) == (observe_state(inv_B), decide_basestock(observe_state(inv_B))) and (state_C, a_C) == (observe_state(inv_C), decide_basestock(observe_state(inv_C)))
    print("CHECK 2 —", "PASS" if ok else "NOT YET")


## Stop 3 — Solve one REAL day

![Real day](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-agent-env.png)
![Promo path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/llm-tool-loop.png)


In [ ]:
# STOP 3a — RUN WITH ME

def pick_real_day(prefer_promo: bool):
    for r in ROWS:
        if r["Is_Promotional"] is prefer_promo:
            return r
    return ROWS[0]

DAY_NORMAL = pick_real_day(False)
DAY_PROMO = pick_real_day(True)
keys = ["Date", "Current_Inventory", "Demand", "Stockout_Units", "Is_Promotional", "Action_Taken", "Order_Quantity", "Profit", "Holding_Cost", "Stockout_Cost"]
print("REAL DAY — normal")
print({k: DAY_NORMAL[k] for k in keys})
print(" state=", observe_state(DAY_NORMAL["Current_Inventory"]), " basestock=", decide_basestock(observe_state(DAY_NORMAL["Current_Inventory"])))
print()
print("REAL DAY — promotional")
print({k: DAY_PROMO[k] for k in keys})
print(" state=", observe_state(DAY_PROMO["Current_Inventory"]), " basestock=", decide_basestock(observe_state(DAY_PROMO["Current_Inventory"])))

PROMO_BRIEF = {
    "date": DAY_PROMO["Date"],
    "subject": f"Electronics campaign live — {DAY_PROMO['Date']}",
    "body": (
        f"Marketing confirms promotional day. demand={DAY_PROMO['Demand']}, "
        f"inventory={DAY_PROMO['Current_Inventory']}, stockout_units={DAY_PROMO['Stockout_Units']}. "
        "Do not auto-change Order_Quantity without desk-lead stamp."
    ),
}
print()
print("EXCEPTION DOCUMENT:")
print(" ", PROMO_BRIEF["subject"])
print(" ", PROMO_BRIEF["body"])



In [ ]:
# STOP 3b — TRY THIS
promo_date = ""  # <<< FILL DAY_PROMO Date
base_a = None    # <<< FILL basestock 0/1/2
needs_override_draft = None  # <<< FILL True/False
email_inside_inventory_number = None  # <<< FILL True/False
print(promo_date, base_a, needs_override_draft, email_inside_inventory_number)
if not promo_date or base_a is None or needs_override_draft is None or email_inside_inventory_number is None:
    print("NOT YET")
else:
    ok = promo_date == DAY_PROMO["Date"] and base_a == decide_basestock(observe_state(DAY_PROMO["Current_Inventory"])) and needs_override_draft is True and email_inside_inventory_number is False
    print("CHECK 3 —", "PASS" if ok else "NOT YET")


## Stop 4 — Family on real situations

![Choose](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/two-families.png)


In [ ]:
# STOP 4 — TRY THIS
# A) Only Current_Inventory → Order/Maintain/Reduce daily; profit costs; no briefs
case_A = ""  # <<< FILL
# B) One campaign brief; human stamp; not daily controller alone
case_B = ""  # <<< FILL
# C) THIS desk: 730 inventory days AND many Is_Promotional days
case_C = ""  # <<< FILL
# D) No signed reward policy; autonomous orders forbidden for now
case_D = ""  # <<< FILL
print(case_A, case_B, case_C, case_D)
ok = [case_A, case_B, case_C, case_D] == ["RL", "LLM", "hybrid", "do not automate"]
print("CHECK 4 —", "PASS" if ok else "NOT YET — expected RL / LLM / hybrid / do not automate")


## Stop 5 — MDP from REAL columns

![MDP](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-mdp.png)


In [ ]:
# STOP 5 — TRY THIS
mdp_states = ""  # <<< FILL mention Current_Inventory
mdp_actions = []  # <<< FILL [0,1,2]
mdp_reward = ""  # <<< FILL profit/sales/margin
mdp_why_repeatable = ""  # <<< FILL
print(mdp_states, mdp_actions, mdp_reward, mdp_why_repeatable)
print("Evidence N_DAYS=", N_DAYS)
if not mdp_states or not mdp_actions or not mdp_reward or not mdp_why_repeatable:
    print("NOT YET")
else:
    ok = mdp_actions == [0, 1, 2] and any(w in mdp_reward.lower() for w in ("margin", "sales", "profit")) and "inventory" in mdp_states.lower()
    print("CHECK 5 —", "PASS" if ok else "NOT YET")


## Stop 6 — LLM on the REAL promo brief

![LLM](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-llm-agent.png)
![Stamp](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-handoff.png)


In [ ]:
# STOP 6a — RUN WITH ME
print(PROMO_BRIEF["subject"])
print(PROMO_BRIEF["body"])
print("Basestock a=", decide_basestock(observe_state(DAY_PROMO["Current_Inventory"])))
print("Historical Action_Taken=", DAY_PROMO["Action_Taken"], "Order_Quantity=", DAY_PROMO["Order_Quantity"])


In [ ]:
# STOP 6b — TRY THIS
llm_goal = ""  # <<< FILL
llm_tools = []  # <<< FILL
llm_knowledge = ""  # <<< FILL
llm_human_handoff = ""  # <<< FILL
proposed_override_a = None  # <<< FILL
proposal_reason = ""  # <<< FILL must include real promo DATE
print(llm_goal, llm_tools, llm_human_handoff, proposed_override_a, proposal_reason)
if not llm_goal or not llm_tools or not llm_human_handoff or proposed_override_a is None or not proposal_reason:
    print("NOT YET")
else:
    ok = proposed_override_a in (0, 1, 2) and DAY_PROMO["Date"] in proposal_reason
    print("CHECK 6 —", "PASS" if ok else "NOT YET — include real DATE")


## Stop 7 — Memo for THIS real desk

![Memo](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)


In [ ]:
# STOP 7a — TRY THIS
say_it = ""  # <<< FILL LLM / RL / hybrid / do not automate
print("Vote:", say_it or "(FILL)", "promo days:", len(PROMO_DAYS), "/", N_DAYS)


In [ ]:
# STOP 7b-c — TRY until PASS
releases_daily_order = ""  # <<< FILL
releases_email_override = ""  # <<< FILL
because_text = ""  # <<< FILL ≥40 chars; mention Electronics or inventory_dataset_full

def reward_ok(text):
    t = (text or "").lower()
    return "margin" in t or "sales" in t or "profit" in t

def review_memo(memo):
    issues = []
    llm, mdp = memo.get("llm_design", {}), memo.get("mdp_design", {})
    rec = (memo.get("recommendation") or "").strip()
    because = (memo.get("because") or "").strip()
    who_order = (memo.get("releases_daily_order") or "").strip()
    who_email = (memo.get("releases_email_override") or "").strip()
    if not (llm.get("goal") or "").strip() or not llm.get("tools") or not (llm.get("human_handoff") or "").strip():
        issues.append("LLM needs goal, tools, human_handoff.")
    if not (mdp.get("states") or "").strip() or not mdp.get("actions") or not reward_ok(mdp.get("reward", "")):
        issues.append("MDP needs states, actions, reward with margin/sales/profit.")
    if not (mdp.get("why_repeatable") or "").strip():
        issues.append("MDP needs why_repeatable.")
    if rec not in {"LLM", "RL", "hybrid", "do not automate"}:
        issues.append("bad recommendation")
    if rec == "LLM" and "inventory" in (mdp.get("states") or "").lower():
        issues.append("inventory state + LLM-only → usually hybrid")
    if len(because) < 40:
        issues.append("because too short")
    if "electronics" not in because.lower() and "inventory_dataset" not in because.lower():
        issues.append("because must mention Electronics or inventory_dataset_full")
    if len(who_order) < 3 or len(who_email) < 3:
        issues.append("name both release owners")
    if issues:
        print("NOT YET");
        [print(" -", i) for i in issues]
        return False
    print("PASS — real-case memo ready")
    print(rec, "|", because)
    print("Order:", who_order, "| Override:", who_email)
    print("Evidence days", N_DAYS, "promo", len(PROMO_DAYS), "promo date", DAY_PROMO["Date"])
    return True

MEMO = {
    "process": "Electronics replenishment exceptions (inventory_dataset_full.csv)",
    "llm_design": {"goal": llm_goal, "tools": list(llm_tools), "knowledge": llm_knowledge, "human_handoff": llm_human_handoff},
    "mdp_design": {"states": mdp_states, "actions": list(mdp_actions), "reward": mdp_reward, "why_repeatable": mdp_why_repeatable},
    "recommendation": (say_it or "").strip(),
    "releases_daily_order": releases_daily_order,
    "releases_email_override": releases_email_override,
    "because": because_text,
}
review_memo(MEMO)


## Stop 8 — LAB on real rows

![Loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/agent-loop.png)


In [ ]:
# CH1 TRY THIS — mean Profit promo vs normal
promo_profit = None  # <<< FILL
normal_profit = None  # <<< FILL
print(promo_profit, normal_profit)
print("Hint: sum Profit where Is_Promotional / len(PROMO_DAYS)")


In [ ]:
# CH2 TRY THIS — worst real stockout day
day = max(ROWS, key=lambda r: r["Stockout_Units"])
print("Worst stockout:", day["Date"], day["Stockout_Units"], "promo=", day["Is_Promotional"])
family_for_recurring_stockout_plus_notes = ""  # <<< FILL
print("CHECK:", "PASS" if family_for_recurring_stockout_plus_notes == "hybrid" else "NOT YET")


In [ ]:
# CH3 TRY THIS — break then fix
bad = dict(MEMO)
bad["recommendation"] = "LLM"
print("Broken:"); review_memo(bad)
print("---"); print("Fixed:"); review_memo(MEMO)


## Stop 9 — Framework line


In [ ]:
# STOP 9 TRY THIS
framework_family = ""  # <<< FILL
framework_line = ""  # <<< FILL
print(framework_family or "(FILL)"); print(framework_line or "(FILL)")


## Raise a hand — real case complete

1. [ ] Explained `Current_Inventory`, `Profit`, `Is_Promotional`  
2. [ ] Loaded real `inventory_dataset_full.csv` Electronics days  
3. [ ] Solved a **real promo date** (base $a$ + override draft)  
4. [ ] Memo PASS names both release owners  
5. [ ] `because` mentions Electronics / `inventory_dataset_full`  
